# Ablações de variáveis em val_ref

Treina cada conjunto de colunas da tabela de ablações e mede o efeito sobre o conjunto P.
Lê as partições `train_ref` e `val_ref` do artefato supervisionado final.
Grava duas tabelas em `outputs/final/`: resultados por linha e ROC-AUC por ano.

In [ ]:
from pathlib import Path
import sys


def encontrar_raiz_projeto(start: Path | None = None) -> Path:
    atual = (start or Path.cwd()).resolve()
    for candidato in (atual, *atual.parents):
        if (candidato / "src").is_dir() and (candidato / "pyproject.toml").is_file():
            return candidato
    raise RuntimeError(f"Raiz do projeto não encontrada a partir de: {atual}")


ROOT_DIR = encontrar_raiz_projeto()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.utils import config as C

In [ ]:
from src.data.supervised_artifact import load_supervised_artifact

df, manifest = load_supervised_artifact(ROOT_DIR, expected_kind="final", partitions=("train_ref", "val_ref"))
print(df.shape, df[manifest["partition_column"]].value_counts().to_dict())

In [ ]:
import gc
import time

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import Pipeline

from src.evaluation.metrics import auc, average_precision, bootstrap_grupo, brier
from src.features.engineering import recodificar_comorbidades_por_fator_risco
from src.models.factory import build_finalista
from src.models.pipeline import build_preprocessor
from src.utils import config as C  # noqa: N812

TRES_SINTOMAS = ["perd_olft", "perd_pala", "fadiga"]
COMORBIDADES = ["cardiopati", "hematologi", "sind_down", "hepatica", "asma", "diabetes", "neurologic",
                "pneumopati", "imunodepre", "renal", "obesidade", "out_morbi", "puerpera"]


def pipeline_ablacao(familia, numericas, categoricas):
    """Pipeline com hiperparâmetros padrão para um conjunto de colunas escolhido."""
    if familia == "logistic":
        return build_finalista("logistic", preprocessor=build_preprocessor(numericas, categoricas))
    mascara = [False] * len(numericas) + [True] * len(categoricas)
    classificador = HistGradientBoostingClassifier(
        learning_rate=0.08, max_iter=60, max_leaf_nodes=31, min_samples_leaf=200, l2_regularization=0.10,
        early_stopping=True, validation_fraction=0.1, n_iter_no_change=10, class_weight="balanced",
        categorical_features=mascara, random_state=C.SEED,
    )
    return Pipeline([("pre", build_preprocessor(numericas, categoricas, saida="ordinal")), ("clf", classificador)])


def executar_ablacoes(df, manifest):
    """Treina em train_ref e mede em val_ref cada linha da tabela de ablações."""
    col_part, alvo = manifest["partition_column"], manifest["target_column"]
    num, cat = list(manifest["numeric_columns"]), list(manifest["categorical_columns"])
    treino, valid = df[col_part] == "train_ref", df[col_part] == "val_ref"
    y_tr, y_va = df.loc[treino, alvo].to_numpy(), df.loc[valid, alvo].to_numpy()
    prevalencia = float(y_va.mean())

    # Verifica se a recodificação tem o que fazer (evita criar df_recod quando não há mudança)
    fator_risc = df["fator_risc"].astype("string").str.strip().str.upper()
    nao = fator_risc.isin(["2", "2.0", "N", "NAO", "NÃO"])
    mudancas_recod = sum(
        int(((df[c].isna()) | (df[c].astype("string").str.strip() == "")).__and__(nao).sum())
        for c in COMORBIDADES
    )
    pular_recod = mudancas_recod == 0
    if pular_recod:
        print(f"P_comorbidades_recodificadas = P (0 mudanças em {len(COMORBIDADES)} comorbidades); linha pulada.")

    linhas_leves = {
        "P": (df, num, cat, ["logistic", "boosting"]),
        "so_idade_sexo": (df, [c for c in num if c == "idade_anos"], [c for c in cat if c == "sexo_categoria"], ["logistic"]),
        "P_mais_3_sintomas": (df, num, cat + TRES_SINTOMAS, ["logistic", "boosting"]),
        "P_menos_gravidade": (df, num, [c for c in cat if c not in ("saturacao", "desc_resp")], ["logistic"]),
        "P_mais_contexto": (df, num, cat + ["cs_escol_n", "vacina"], ["logistic"]),
        "P_mais_regiao": (df, num, cat + ["regiao_residencia"], ["logistic"]),
    }

    resultados, probabilidades = [], {}

    def rodar(linha, quadro, n_cols, c_cols, familia):
        t0 = time.perf_counter()
        X_tr = quadro.loc[treino, n_cols + c_cols]
        X_va = quadro.loc[valid, n_cols + c_cols]
        pipe = pipeline_ablacao(familia, n_cols, c_cols)
        pipe.fit(X_tr, y_tr)
        prob = pipe.predict_proba(X_va)[:, 1].astype("float32")
        del X_tr, X_va, pipe
        gc.collect()
        probabilidades[(linha, familia)] = prob
        pr = average_precision(y_va, prob)
        resultados.append({"linha": linha, "familia": familia, "n_colunas": len(n_cols + c_cols),
                           "pr_auc": pr, "pr_auc_sobre_prevalencia": pr / prevalencia,
                           "roc_auc": auc(y_va, prob), "brier": brier(y_va, prob),
                           "duracao_s": round(time.perf_counter() - t0, 1)})

    # 1) as 5 ablações leves, na base original
    for linha, (quadro, n_cols, c_cols, familias) in linhas_leves.items():
        for familia in familias:
            rodar(linha, quadro, n_cols, c_cols, familia)

    # 2) P_comorbidades_recodificadas só no fim, e liberando df_recod em seguida
    if not pular_recod:
        base = df[COMORBIDADES + ["fator_risc"]].astype("object")
        recod = recodificar_comorbidades_por_fator_risco(base)
        del base
        df_recod = df.drop(columns=COMORBIDADES).join(recod[COMORBIDADES])
        del recod
        gc.collect()
        for familia in ["logistic", "boosting"]:
            rodar("P_comorbidades_recodificadas", df_recod, num, cat, familia)
        del df_recod
        gc.collect()

    tabela = pd.DataFrame(resultados)
    ref = tabela[tabela["linha"] == "P"].set_index("familia")
    tabela["dif_pr_auc_vs_P"] = tabela.apply(lambda r: r["pr_auc"] - ref.loc[r["familia"], "pr_auc"], axis=1)
    tabela["dif_roc_auc_vs_P"] = tabela.apply(lambda r: r["roc_auc"] - ref.loc[r["familia"], "roc_auc"], axis=1)

    anos = df.loc[valid, "ano_internacao"].to_numpy()
    grupos = df.loc[valid, "group_id"].to_numpy()
    por_ano = []
    for ano in sorted(set(anos)):
        m = anos == ano
        linha_ano = {"ano": int(ano), "n": int(m.sum()), "obitos": int(y_va[m].sum())}
        for familia in ("logistic", "boosting"):
            p_p, p_3 = probabilidades[("P", familia)][m], probabilidades[("P_mais_3_sintomas", familia)][m]
            linha_ano[f"roc_auc_P_{familia}"] = auc(y_va[m], p_p)
            linha_ano[f"roc_auc_mais3_{familia}"] = auc(y_va[m], p_3)
            ic = bootstrap_grupo(y_va[m], p_3, grupos[m], metrica="auc", prob_b=p_p, n=300)
            linha_ano[f"dif_mais3_menos_P_{familia}"] = ic["estimativa_diferenca"]
            linha_ano[f"ic_inf_{familia}"], linha_ano[f"ic_sup_{familia}"] = ic["ic_inferior"], ic["ic_superior"]
        por_ano.append(linha_ano)
    return tabela, pd.DataFrame(por_ano)

In [ ]:
from IPython.display import display

tabela, por_ano = executar_ablacoes(df, manifest)
display(tabela.round(4))
display(por_ano.round(4))

In [ ]:
SAIDA_DIR = ROOT_DIR / "outputs" / "final"
SAIDA_DIR.mkdir(parents=True, exist_ok=True)

tabela.to_csv(SAIDA_DIR / "ablacoes_val_ref.csv", index=False)
por_ano.to_csv(SAIDA_DIR / "ablacoes_roc_por_ano.csv", index=False)

## Decisão proposta

**Três sintomas (perd_olft, perd_pala, fadiga):** manter/retirar, porque …

**Comorbidades recodificadas:** manter/recodificar, porque …